In [19]:
# Biblioteca

import polars as pl
import numpy as np
import os

# Localização dos arquivos

pasta = '/home/orfei/mestrado/progs/array_10x10'

# Retorna o nome do arquivo formatado
import re
def formatar_string(s):
   
    match = re.match(r'([a-zA-Z]+)([0-9]+(?:\.[0-9]+)?[Ee][0-9]+)', s)

    if match:
        
        palavra = match.group(1).capitalize()
        numero = match.group(2)

        return f'{palavra} {numero}'
    else:
        return s


dataframes = []

for arquivo in os.listdir(pasta):
    if arquivo.endswith('_array'):  
        print(f'Processando arquivo: {arquivo}') 
        # Leitura do arquivo
        try:
            df = pl.read_csv(os.path.join(pasta, arquivo), has_header=False).filter(pl.col('column_1').str.contains('TRIG'))
            df = df.with_columns(pl.col("column_1").str.split(" ").alias("split_column"))
            df = df.with_columns(pl.col("split_column").list.get(0).alias("TRIG"))
            df = df.with_columns(pl.col("split_column").list.get(1).cast(pl.Int64).alias("count"))
            
            # Nome formatado do arquivo
            name = formatar_string(arquivo)
            
            # Transformação das contagens de partículas em listas e na média de contagens
            df = df.group_by('TRIG').agg(pl.col("count")).sort(pl.col("TRIG").str.extract(r"TRIG([0-9]*)", 1).cast(int))
            df = df.with_columns(density=pl.col('count').list.mean())
            df = df.with_columns(
                pl.lit(name.split()[0]).alias('composition'),
                pl.lit(name.split()[1]).alias('energy')
            )
            df = df.drop('count')
            
            # Adiciona o dataframe à lista
            dataframes.append(df)
        except Exception as e:
            print(f'Erro ao processar o arquivo {arquivo}: {e}') 


# Verifica se a lista de dataframes não está vazia
if dataframes:
    # Combina todos os dataframes em um único dataframe
    df_final = pl.concat(dataframes)
    # Embaralha o dataframe
    df_final = df_final.sample(fraction = 1.0, shuffle = True)
    # Exibe o dataframe final
    print(df_final)
else:
    print("Nenhum dataframe foi criado. Verifique se há arquivos válidos na pasta e se os arquivos estão no formato correto.")

Processando arquivo: proton1E15_array
Processando arquivo: iron4.64E15_array
Processando arquivo: iron1E14_array
Processando arquivo: proton4.64E15_array
Processando arquivo: photon4.64E14_array
Processando arquivo: photon2.15E15_array
Processando arquivo: photon4.64E15_array
Processando arquivo: proton2.15E14_array
Processando arquivo: iron2.15E15_array
Processando arquivo: photon1E15_array
Processando arquivo: photon1E14_array
Processando arquivo: iron1E15_array
Processando arquivo: proton1E13_array
Processando arquivo: proton1E14_array
Processando arquivo: photon1E13_array
Processando arquivo: proton2.15E15_array
Processando arquivo: iron2.15E14_array
Processando arquivo: iron4.64E14_array
Processando arquivo: photon2.15E14_array
Processando arquivo: proton4.64E14_array
shape: (2_000, 4)
┌────────┬─────────┬─────────────┬─────────┐
│ TRIG   ┆ density ┆ composition ┆ energy  │
│ ---    ┆ ---     ┆ ---         ┆ ---     │
│ str    ┆ f64     ┆ str         ┆ str     │
╞════════╪════════

In [33]:
# Normalização da densidade
density_min = df_final.select(pl.col('density').min()).to_numpy()[0, 0]
density_max = df_final.select(pl.col('density').max()).to_numpy()[0, 0]
    
df_normalized = df_final.with_columns(
      ((pl.col('density') - density_min) / (density_max - density_min)).alias('density_normalized')
    )
df_normalized

TRIG,density,composition,energy,density_normalized
str,f64,str,str,f64
"""TRIG83""",0.19,"""Iron""","""4.64E14""",0.000719
"""TRIG27""",0.9,"""Iron""","""1E15""",0.003407
"""TRIG96""",0.0,"""Proton""","""1E13""",0.0
"""TRIG35""",93.15,"""Photon""","""4.64E15""",0.352614
"""TRIG98""",0.15,"""Photon""","""2.15E14""",0.000568
"""TRIG38""",2.71,"""Iron""","""2.15E15""",0.010259
"""TRIG31""",0.5,"""Photon""","""4.64E14""",0.001893
"""TRIG93""",0.16,"""Photon""","""2.15E14""",0.000606
"""TRIG81""",0.3,"""Iron""","""1E15""",0.001136
